# SphereTok: Training an Egocentric 3D World Tokenizer on Kaggle GPU
**Paper:** *Beyond the Airport Tower: Egocentric 3D World Tokenizers for Embodied Spatial Intelligence in Minecraft*

### Overview
This notebook trains the **SphereTok** 3D Vector-Quantized Variational Autoencoder (3D VQ-VAE) on procedural 3D Minecraft voxel chunks using GPU acceleration.
- **Input:** `(B, 32, 32, 32)` egocentric voxel volume
- **Compression:** `4x4x4` micro-cube patchification + Sparse air pruning + 512-codebook quantization
- **Output:** Discrete 3D spatial tokens + continuous tokens for Transformer backbones


In [ ]:
# 1. Environment & GPU Verification
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('Note: To enable GPU on Kaggle, go to Settings -> Accelerator -> GPU T4 x2')


## 2. Procedural Minecraft Voxel Chunk Dataset
Generates rich 3D voxel chunks containing realistic Minecraft features: undulating terrain, underground cave networks, parkour gap structures, and vegetation.


In [ ]:
class ProceduralMinecraftDataset(torch.utils.data.Dataset):
    def __init__(self, num_samples=2000, grid_size=32):
        self.num_samples = num_samples
        self.grid_size = grid_size

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # 0: Air, 1: Stone, 2: Dirt/Grass, 3: Wood, 4: Water
        grid = torch.zeros((self.grid_size, self.grid_size, self.grid_size), dtype=torch.long)
        
        # Random ground height baseline
        base_h = np.random.randint(8, 14)
        
        # Generate undulating terrain with low-frequency waves
        x_coords = np.linspace(0, 2 * np.pi, self.grid_size)
        z_coords = np.linspace(0, 2 * np.pi, self.grid_size)
        X, Z = np.meshgrid(x_coords, z_coords)
        terrain_height = (base_h + 3 * np.sin(X + np.random.uniform(0, 3)) + 2 * np.cos(Z + np.random.uniform(0, 3))).astype(int)
        terrain_height = np.clip(terrain_height, 4, 22)

        for x in range(self.grid_size):
            for z in range(self.grid_size):
                h = terrain_height[x, z]
                grid[0:max(1, h-2), x, z] = 1 # Stone base
                grid[max(0, h-2):h, x, z] = 2 # Surface soil/grass

        # Carve random underground cave cavities
        if np.random.rand() > 0.3:
            cave_x, cave_y, cave_z = np.random.randint(8, 24, size=3)
            cave_r = np.random.randint(3, 6)
            for y in range(max(1, cave_y - cave_r), min(self.grid_size, cave_y + cave_r)):
                for x in range(max(0, cave_x - cave_r), min(self.grid_size, cave_x + cave_r)):
                    for z in range(max(0, cave_z - cave_r), min(self.grid_size, cave_z + cave_r)):
                        if (x - cave_x)**2 + (y - cave_y)**2 + (z - cave_z)**2 <= cave_r**2:
                            grid[y, x, z] = 0 # Air inside cave

        # Add parkour pillars or structures
        if np.random.rand() > 0.4:
            px, pz = np.random.randint(6, 26, size=2)
            pillar_h = np.random.randint(3, 7)
            ground_y = terrain_height[px, pz]
            grid[ground_y:min(self.grid_size, ground_y + pillar_h), px, pz] = 3 # Wood column

        return grid

print('Procedural Minecraft dataset generator defined successfully.')


## 3. SphereTok Architecture (3D VQ-VAE + Sparse Pruning)


In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=512, embedding_dim=128, beta=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.beta = beta
        self.embedding = nn.Embedding(self.num_embeddings, self.embedding_dim)
        self.embedding.weight.data.uniform_(-1.0 / self.num_embeddings, 1.0 / self.num_embeddings)

    def forward(self, z):
        z_flat = z.reshape(-1, self.embedding_dim)
        distances = (
            torch.sum(z_flat ** 2, dim=1, keepdim=True)
            + torch.sum(self.embedding.weight ** 2, dim=1)
            - 2 * torch.matmul(z_flat, self.embedding.weight.t())
        )
        indices = torch.argmin(distances, dim=1)
        z_q = self.embedding(indices).view(z.shape)
        loss_codebook = F.mse_loss(z_q, z.detach())
        loss_commitment = F.mse_loss(z_q.detach(), z)
        vq_loss = loss_codebook + self.beta * loss_commitment
        z_q = z + (z_q - z).detach()
        return z_q, vq_loss, indices.view(z.shape[0], z.shape[1])

class EgocentricPositionalEncoding(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(3, embedding_dim // 2),
            nn.SiLU(),
            nn.Linear(embedding_dim // 2, embedding_dim)
        )

    def forward(self, patch_coords, yaw=0.0, pitch=0.0):
        dx, dy, dz = patch_coords[..., 0], patch_coords[..., 1], patch_coords[..., 2]
        r = torch.sqrt(dx**2 + dy**2 + dz**2 + 1e-6)
        yaw_rel = torch.atan2(dz, dx) - yaw
        pitch_rel = torch.asin(torch.clamp(dy / r, -0.999, 0.999)) - pitch
        spherical = torch.stack([r / 16.0, yaw_rel / math.pi, pitch_rel / (math.pi / 2.0)], dim=-1)
        return self.mlp(spherical)

class PatchEncoder3D(nn.Module):
    def __init__(self, in_channels=16, latent_dim=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_channels, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.SiLU(),
            nn.Conv3d(64, latent_dim, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(latent_dim),
            nn.SiLU(),
            nn.AdaptiveAvgPool3d(1)
        )
    def forward(self, x):
        return self.conv(x).flatten(1)

class PatchDecoder3D(nn.Module):
    def __init__(self, out_channels=16, latent_dim=128):
        super().__init__()
        self.proj = nn.Linear(latent_dim, 64 * 2 * 2 * 2)
        self.deconv = nn.Sequential(
            nn.ConvTranspose3d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.SiLU(),
            nn.ConvTranspose3d(64, out_channels, kernel_size=4, stride=2, padding=1)
        )
    def forward(self, z):
        x = self.proj(z).view(-1, 64, 2, 2, 2)
        return self.deconv(x)

class SphereTok(nn.Module):
    def __init__(self, num_block_classes=16, grid_size=32, patch_size=4, latent_dim=128, codebook_size=512):
        super().__init__()
        self.num_block_classes = num_block_classes
        self.grid_size = grid_size
        self.patch_size = patch_size
        self.latent_dim = latent_dim
        self.num_patches_axis = grid_size // patch_size
        self.total_patches = self.num_patches_axis ** 3

        self.block_embed = nn.Embedding(num_block_classes, num_block_classes)
        self.block_embed.weight.data = torch.eye(num_block_classes)
        self.block_embed.weight.requires_grad = False

        self.encoder = PatchEncoder3D(in_channels=num_block_classes, latent_dim=latent_dim)
        self.vq = VectorQuantizer(num_embeddings=codebook_size, embedding_dim=latent_dim)
        self.pos_encoder = EgocentricPositionalEncoding(embedding_dim=latent_dim)
        self.decoder = PatchDecoder3D(out_channels=num_block_classes, latent_dim=latent_dim)

        coords = torch.arange(self.num_patches_axis) * patch_size + (patch_size / 2.0) - (grid_size / 2.0)
        gx, gy, gz = torch.meshgrid(coords, coords, coords, indexing='ij')
        self.register_buffer('patch_centers', torch.stack([gx.flatten(), gy.flatten(), gz.flatten()], dim=-1))

    def extract_patches(self, grid):
        B, D, H, W = grid.shape
        P = self.patch_size
        N = self.num_patches_axis
        patches = grid.view(B, N, P, N, P, N, P).permute(0, 1, 3, 5, 2, 4, 6).contiguous()
        return patches.view(B, N * N * N, P, P, P)

    def forward(self, voxel_grid, yaw=0.0, pitch=0.0):
        B = voxel_grid.shape[0]
        patches = self.extract_patches(voxel_grid)
        flat_patches = patches.view(-1, self.patch_size, self.patch_size, self.patch_size)
        one_hot = self.block_embed(flat_patches).permute(0, 4, 1, 2, 3)
        
        z = self.encoder(one_hot).view(B, self.total_patches, self.latent_dim)
        z_q, vq_loss, token_indices = self.vq(z)
        
        centers = self.patch_centers.unsqueeze(0).expand(B, -1, -1)
        pos_emb = self.pos_encoder(centers, yaw=yaw, pitch=pitch)
        continuous_tokens = z_q + pos_emb
        
        recon = self.decoder(z_q.view(-1, self.latent_dim))
        recon_loss = F.cross_entropy(recon, flat_patches)
        total_loss = recon_loss + vq_loss
        
        return {
            'tokens': token_indices,
            'continuous_tokens': continuous_tokens,
            'recon_loss': recon_loss,
            'vq_loss': vq_loss,
            'total_loss': total_loss
        }

print('SphereTok model architecture initialized.')


## 4. GPU Training Loop
Trains SphereTok on 2,000 Minecraft procedural chunks with AdamW and Cosine Annealing scheduler.


In [ ]:
BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-3

dataset = ProceduralMinecraftDataset(num_samples=1600)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

model = SphereTok(num_block_classes=16, codebook_size=512, latent_dim=128).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f'Starting GPU Training on {device} ({len(dataset)} samples, {EPOCHS} epochs)...')
history = {'total': [], 'recon': [], 'vq': []}

model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss, epoch_recon, epoch_vq = 0.0, 0.0, 0.0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = out['total_loss']
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_recon += out['recon_loss'].item()
        epoch_vq += out['vq_loss'].item()
    
    scheduler.step()
    n_batches = len(loader)
    avg_loss = epoch_loss / n_batches
    avg_recon = epoch_recon / n_batches
    avg_vq = epoch_vq / n_batches
    
    history['total'].append(avg_loss)
    history['recon'].append(avg_recon)
    history['vq'].append(avg_vq)
    
    print(f'Epoch [{epoch:02d}/{EPOCHS:02d}] | Total: {avg_loss:.4f} | Recon: {avg_recon:.4f} | VQ: {avg_vq:.4f}')

print('\nTraining complete!')
torch.save(model.state_dict(), 'spheretok_minecraft_checkpoint.pt')
print('Model saved to spheretok_minecraft_checkpoint.pt')


## 5. Visualizing Convergence & Checkpoint Download


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history['total'], label='Total Loss', color='black', linewidth=2)
plt.plot(history['recon'], label='Reconstruction Loss', color='royalblue', linestyle='--')
plt.plot(history['vq'], label='Codebook VQ Loss', color='forestgreen', linestyle=':')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('SphereTok 3D VQ-VAE Training Convergence on Minecraft Voxel Data')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('spheretok_training_curve.png', dpi=300)
plt.show()
print('Saved plot to spheretok_training_curve.png')
